<a href="https://colab.research.google.com/github/MiguelCortezPino/etl-data-pipeline-1701472022/blob/main/notebooks/E_movimientos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd


In [2]:
url = "https://raw.githubusercontent.com/MiguelCortezPino/etl-data-pipeline-1701472022/refs/heads/main/data/raw/E_movimientos.csv"
df = pd.read_csv(url)

df.head()

,id_inventario,id_bodega,item,cantidad,costo_unitario
0,INV5000,BOD103,Producto 45,359,192.81
1,INV5001,BOD111,Producto 2,419,153.48
2,INV5002,BOD999,Producto 120,58,104.91
3,INV5003,BOD100,Producto 42,422,$ 138.66
4,INV5004,BOD112,Producto 79,257,NaN


In [3]:
#Exploracion de datos
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_inventario   166 non-null    object
 1   id_bodega       161 non-null    object
 2   item            166 non-null    object
 3   cantidad        161 non-null    object
 4   costo_unitario  161 non-null    object
dtypes: object(5)
memory usage: 6.6+ KB


,0
id_inventario,0
id_bodega,5
item,0
cantidad,5
costo_unitario,5


In [9]:
#limpiesza de datos
df['costo_unitario'] = pd.to_numeric(df['costo_unitario'].astype(str).str.replace('$', '', regex=False).str.replace(',', ''), errors='coerce')
df['cantidad'] = pd.to_numeric(df['cantidad'].astype(str).str.replace(',', ''), errors='coerce')

print(df.info())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 166 entries, 0 to 165
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_inventario   166 non-null    object 
 1   id_bodega       161 non-null    object 
 2   item            166 non-null    object 
 3   cantidad        154 non-null    float64
 4   costo_unitario  155 non-null    float64
dtypes: float64(2), object(3)
memory usage: 6.6+ KB
None
id_inventario      0
id_bodega          5
item               0
cantidad          12
costo_unitario    11
dtype: int64


In [7]:
display(df.head())

,id_inventario,id_bodega,item,cantidad,costo_unitario
0,INV5000,BOD103,Producto 45,359.0,192.81
1,INV5001,BOD111,Producto 2,419.0,153.48
2,INV5002,BOD999,Producto 120,58.0,104.91
3,INV5003,BOD100,Producto 42,422.0,138.66
4,INV5004,BOD112,Producto 79,257.0,NaN


In [ ]:
#motivo de rechazo



In [15]:
#motivo de rechazo

def generate_rejection_reason(row):
    reasons = []
    if pd.isna(row['id_bodega']):
        reasons.append('id_bodega_missing')
    if pd.isna(row['cantidad']):
        reasons.append('cantidad_missing')
    if pd.isna(row['costo_unitario']):
        reasons.append('costo_unitario_missing')

    if not reasons:
        return 'No rechazo'
    else:
        return ', '.join(reasons)

df['motivo_rechazo'] = df.apply(generate_rejection_reason, axis=1)

print(df['motivo_rechazo'].value_counts())
display(df.head())

motivo_rechazo
No rechazo                                  139
cantidad_missing                             11
costo_unitario_missing                       10
id_bodega_missing                             5
cantidad_missing, costo_unitario_missing      1
Name: count, dtype: int64


,id_inventario,id_bodega,item,cantidad,costo_unitario,motivo_rechazo
0,INV5000,BOD103,Producto 45,359.0,192.81,No rechazo
1,INV5001,BOD111,Producto 2,419.0,153.48,No rechazo
2,INV5002,BOD999,Producto 120,58.0,104.91,No rechazo
3,INV5003,BOD100,Producto 42,422.0,138.66,No rechazo
4,INV5004,BOD112,Producto 79,257.0,NaN,costo_unitario_missing


In [16]:
#creacion de archivos csv reject y curated

# Definir las columnas críticas para la validación
critical_columns = ['id_inventario', 'id_bodega', 'cantidad', 'costo_unitario']

# Identificar filas con valores nulos en alguna de las columnas críticas
df_rechazados = df[df[critical_columns].isnull().any(axis=1)].copy()

# Identificar filas sin valores nulos en ninguna de las columnas críticas
df_validos = df[~df[critical_columns].isnull().any(axis=1)].copy()

In [22]:
# Guardar los DataFrames en archivos CSV
df_validos.to_csv("E_Movimientos_cureted.csv", index=False)
df_rechazados.to_csv("E_Movimientos_rejects.csv", index=False)

print("Archivos guardados con éxito: 'E_movimientos_cureted.csv' y 'E_Movientos_rejects.csv'")

Archivos guardados con éxito: 'E_movimientos_cureted.csv' y 'E_Movientos_rejects.csv'
